In [ ]:
from pathlib import Path
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

from src.datasets.multimodal_feature_dataset import MultiModalFeatureDataset
from src.models.multimodal.multimodal_pvd_classifier import MultimodalPVDClassifier

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Config

In [11]:
IMAGE_FEATURE_PATH = PROJECT_ROOT / "data" / "features" / "encoded_image" / "train_original_image_features_convnext_cbam.npy"
TEXT_FEATURE_PATH = PROJECT_ROOT / "data" / "features" / "encoded_text" / "train_text_embeddings_clip_vitl14.npy"
METADATA_PATH = PROJECT_ROOT / "data" / "features" / "encoded_image" / "train_original_image_features_convnext_cbam_metadata.json"

NUM_CLASSES = 28
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-4

Dataset

In [12]:
dataset = MultiModalFeatureDataset(
    image_feature_path=IMAGE_FEATURE_PATH,
    text_feature_path=TEXT_FEATURE_PATH,
    metadata_path=METADATA_PATH
)

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

Model

In [13]:
model = MultimodalPVDClassifier(
    image_input_dim=1024,
    text_input_dim=768,
    proj_dim=512,
    proj_hidden_dim=768,
    pvd_hidden_dim=768,
    cls_hidden_dim=1024,
    num_classes=NUM_CLASSES,
    dropout=0.3,
    normalize_projection=False
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

Train loop

In [14]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        image_feat = batch["image_feat"].to(device)
        text_feat = batch["text_feat"].to(device)
        labels = batch["label"].to(device)

        logits = model(image_feat, text_feat)   # [B, num_classes]
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = total_loss / total
    epoch_acc = correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")

Epoch [1/20] - Loss: 2.6699 - Acc: 0.2928
Epoch [2/20] - Loss: 1.3746 - Acc: 0.5839
Epoch [3/20] - Loss: 0.9844 - Acc: 0.6832
Epoch [4/20] - Loss: 0.8109 - Acc: 0.7196
Epoch [5/20] - Loss: 0.6824 - Acc: 0.7714
Epoch [6/20] - Loss: 0.6110 - Acc: 0.7954
Epoch [7/20] - Loss: 0.5129 - Acc: 0.8249
Epoch [8/20] - Loss: 0.4392 - Acc: 0.8510
Epoch [9/20] - Loss: 0.3928 - Acc: 0.8677
Epoch [10/20] - Loss: 0.3392 - Acc: 0.8926
Epoch [11/20] - Loss: 0.2765 - Acc: 0.9118
Epoch [12/20] - Loss: 0.2379 - Acc: 0.9208
Epoch [13/20] - Loss: 0.2051 - Acc: 0.9298
Epoch [14/20] - Loss: 0.1750 - Acc: 0.9473
Epoch [15/20] - Loss: 0.1587 - Acc: 0.9448
Epoch [16/20] - Loss: 0.1461 - Acc: 0.9542
Epoch [17/20] - Loss: 0.1258 - Acc: 0.9640
Epoch [18/20] - Loss: 0.1115 - Acc: 0.9666
Epoch [19/20] - Loss: 0.1199 - Acc: 0.9645
Epoch [20/20] - Loss: 0.0915 - Acc: 0.9752


In [15]:
SAVE_DIR = PROJECT_ROOT / "archive" / "multimodal"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = SAVE_DIR / "multimodal_pvd_classifier_state_dict.pth"

torch.save(model.state_dict(), MODEL_PATH)
print("Saved model to:", MODEL_PATH)

Saved model to: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal\multimodal_pvd_classifier_state_dict.pth
